# Hello Bayes!

In [ ]:
%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt

## Recall some theory

- Let $H$ be the hypothesis, let $E$ be the evidence or data, then recall that:

$$
f(H|E) = \frac{f(E|H) f(H)}{f(E)}
$$

- Specifically for an unknown parameter $\theta$ and
- A sample with data $x_1,\ldots,x_n$,
- We have

$$\begin{aligned}
f(\theta|x_1,\ldots,x_n) &=
\frac{f(\theta, x_1, \ldots, x_n)}{f(x_1, \ldots, x_n)} \\
&= \frac{p(\theta)f(x_1, \ldots, x_n|\theta)}{\int f(x_1, \ldots,
x_n|\theta) p(\theta) d\theta}
\end{aligned}$$

## Problem

* Consider a coin which may be biased or not.
* We want to assess the parameter, $p$ but using a Bayesian approach.
* $p$ is our unknown $\theta$
* Want to estimate the distribution of $p$ from an initial guess/belief and data

## Generating the data

In [ ]:
N = 50
data = np.random.choice([0, 1], size=N)

## Setting up the problem

* Assume a prior, we start with a uniform distribution
* We need the likelihood function: $f(x| \theta) = \theta^x (1-\theta)^{1-x}$
* Samples are independent so we can easily construct the joint likelihood.
* We solve the problem by numerical integration
* Recall that we are looking to find $\theta$
    * Fix the domain to 100 points between 0 and 1.

## Back to the problem at hand

In [ ]:
nt = 100
d_theta = 1.0/nt
thetas = np.arange(d_theta/2, 1, d_theta)

In [ ]:
def likelihood(theta, data):
    """Assumes that theta and data are scalars"""
    theta**data*(1-theta)**(1-data)

def joint_likelihood(theta, data):
    """Assumes that theta is a scalar, data is an array."""
    return np.prod(np.pow(theta, data)*np.pow(1-theta, 1-data))

def prior_func(theta):
    if (theta < 0.0) or (theta > 1.0):
        return 0.0
    else:
        return 1.0

In [ ]:
lh = []
for t in thetas:
    lh.append(joint_likelihood(t, data))
lh = np.array(lh)

In [ ]:
def integral_dnr(l, prior, dtheta):
    """Assumes that l and prior are uniformly discretized"""
    return np.sum(l*prior)*dtheta



## Aside: list comprehensions

In [ ]:
[i*i for i in range(5)]

In [ ]:
[(i, i*i) for i in range(5)]

In [ ]:
[i*i for i in range(5) if i%2 == 0]

## Back to the problem

In [ ]:
lh = np.array([joint_likelihood(t, data) for t in thetas])

In [ ]:
prior = np.asarray([prior_func(t) for t in thetas])

In [ ]:
dnr = integral_dnr(lh, prior, d_theta)

In [ ]:
posterior = prior*lh/dnr

In [ ]:
plt.plot(thetas, posterior)

## Textbook answer

- $x=\sum_i x_i$

$$f(\theta|x_1,\ldots,x_n) = \frac{(n+1)! \theta^x (1-\theta)^{n-x}}{x! (n-x)!}$$


In [ ]:
from scipy.special import factorial

def f_exact(theta, x, n):
    return factorial(n+1)*theta**x*(1-theta)**(n-x)/(factorial(x)*factorial(n-x))

In [ ]:
plt.plot(thetas, posterior, label='Computed')
x = np.sum(data)
plt.plot(thetas, f_exact(thetas, x, N), label='Exact')
plt.legend();

## Bayes estimator

- Best estimate of $\theta$ is the mean of the posterior:

$$E[\theta|X_1=x_1, \ldots, X_n=x_n] = \int \theta f(\theta|x_1,\ldots,x_n)
d\theta$$

In [ ]:
# Homework

## Textbook answer

$$E[\theta|X_1=x_1, \ldots, X_n=x_n] = \frac{x+1}{n+2}$$

## Cleaning up

In [ ]:
def calc_posterior(theta, data, prior, likelihood_f):
    """Assumes that theta is uniformly spaced.
    `prior` is the current
    """
    lh = np.array([likelihood_f(t, data) for t in theta])
    d_theta = theta[1] - theta[0]
    dnr = integral_dnr(lh, prior, d_theta)
    post = lh*prior/dnr
    return post

In [ ]:
prior = np.asarray([prior_func(t) for t in thetas])
posterior = calc_posterior(thetas, data, prior, joint_likelihood)

In [ ]:
plt.plot(thetas, posterior)
plt.plot(thetas, prior);

## Incremental updates of posterior

- Can add data to update the posterior

In [ ]:
data = np.random.choice([0, 1], size=5)

In [ ]:
prior = posterior
post = calc_posterior(thetas, data, prior, joint_likelihood)

In [ ]:
plt.plot(thetas, post)
plt.plot(thetas, posterior);

## Making this go faster

- Eliminate looping and use vectorization

In [ ]:
@np.vectorize
def prior_func(theta):
    if (theta < 0.0) or (theta > 1.0):
        return 0.0
    else:
        return 1.0

Use broadcasting instead of looping

In [ ]:
# This is what we had done first
data = np.random.choice([0, 1], size=5)
theta = 0.1
np.prod(np.pow(theta, data)*np.pow(1-theta, 1-data))

What if `theta` is a vector?

In [ ]:
theta = np.array([0.1, 0.2])
btheta = theta[:, None]
print(btheta.shape)
print(btheta)

In [ ]:
(btheta*data).shape

So now we can do this:

In [ ]:
np.prod(np.pow(btheta, data)*np.pow(1 - btheta, 1-data), axis=1)

## Using this for calculation

In [ ]:
def likelihood(theta, data):
    btheta = theta[:, None]
    return np.prod(np.pow(btheta, data)*np.pow(1 - btheta, 1 - data), axis=1)

In [ ]:
def calc_posterior(theta, data, prior, likelihood):
    """Assumes that theta is uniformly spaced."""
    lh = likelihood(theta, data)
    d_theta = theta[1] - theta[0]
    dnr = integral_dnr(lh, prior, d_theta)
    post = lh*prior/dnr
    return post

In [ ]:
posterior = calc_posterior(thetas, data, prior_func(thetas), likelihood)

In [ ]:
plt.plot(thetas, posterior);